# QLoRA fine-tune Qwen2.5-7B trên synthetic data (Colab)
Fine-tune LLM extractor bằng `data/synthetic/train_sft.jsonl` (self-host ≤9B, KHÔNG API ngoài).
Runtime → GPU (T4 chạy được 4-bit; **L4/A100 nhanh hơn nhiều**). Lưu adapter vào Drive vì Colab hay ngắt.

**Quy trình:** cài env → smoke-train (ít bước) kiểm OOM → train full → trỏ config → đo dev so với few-shot.

In [1]:
# 1) Drive: cache model + LƯU ADAPTER bền vững (Colab ngắt là mất /content)
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
ADAPTER = '/content/drive/MyDrive/models/qwen7b-lora'   # adapter lưu ở Drive
print('HF_HOME =', os.environ['HF_HOME'], '| ADAPTER =', ADAPTER)

Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/hf_cache | ADAPTER = /content/drive/MyDrive/models/qwen7b-lora


In [2]:
# 2) Lấy code
%cd /content
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /content/viettel-medreason
!git pull -q

/content
Cloning into 'viettel-medreason'...
remote: Enumerating objects: 576, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 576 (delta 16), reused 9 (delta 4), pack-reused 534 (from 1)
Receiving objects: 100% (576/576), 3.86 MiB | 8.82 MiB/s, done.
Resolving deltas: 100% (235/235), done.
/content/viettel-medreason


In [3]:
# 3) Cài env — KHÔNG cài lại transformers; core + update bitsandbytes/peft/accelerate
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import torch, transformers, bitsandbytes as bnb, peft
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| bnb', bnb.__version__, '| peft', peft.__version__,
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
torch 2.11.0+cu128 | transformers 5.12.1 | bnb 0.49.2 | peft 0.19.1 | GPU Tesla T4


In [4]:
# 4) (Tùy chọn) sinh lại synthetic data — đã ship sẵn trong repo, chỉ chạy nếu muốn tái tạo
# !python src/datagen/gen_synthetic.py --n 1500 --seed 42
!wc -l data/synthetic/train_sft.jsonl

1500 data/synthetic/train_sft.jsonl


In [5]:
# 5) SMOKE-TRAIN: chạy vài chục bước để chắc KHÔNG OOM + đường ống train chạy được
#    (epochs nhỏ; nếu T4 OOM -> giảm --max-len 1536, hoặc dùng --model Qwen/Qwen2.5-3B-Instruct)
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl \
  --out /content/smoke-lora --epochs 0.05 --max-len 2048 --seed 42

config.json: 100% 663/663 [00:00<00:00, 2.51MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.5MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 77.7MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 57.9MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 38.2MB/s]
[data] nạp data/synthetic/train_sft.jsonl ...
[data] 1500 mẫu train
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 108MB/s]
Fetching 4 files: 100% 4/4 [04:07<00:00, 61.94s/it] 
Download complete: 100% 15.2G/15.2G [04:08<00:00, 61.4MB/s]
Loading weights:   1% 2/339 [00:10<31:29,  5.61s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [01:34<00:00,  3.58it/s]
generation_config.json: 100% 243/243 [00:00<00:0

In [6]:
# 6) TRAIN FULL -> lưu adapter thẳng vào Drive (ADAPTER). ~vài giờ trên T4; nhanh hơn trên L4/A100.
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl \
  --out "$ADAPTER" --epochs 2 --batch 1 --grad-accum 16 --lr 1e-4 --max-len 2048 --seed 42
!ls -la "$ADAPTER"

[data] nạp data/synthetic/train_sft.jsonl ...
[data] 1500 mẫu train
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:27<1:09:33, 12.39s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [02:07<00:00,  2.67it/s]
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
  0% 0/188 [00:00<?, ?it/s]Traceback (most recent call last):
  File "/content/viettel-medreason/scripts/train_qlora.py", line 143, in <module>
    main()
  File "/content/viettel-medreason/scripts/train_qlora.py", line 131, in main
    trainer.train()
  File "/usr/local/lib/python3.12/dist-packages/transformers/trainer.py", 

In [7]:
# 7) Trỏ config sang adapter + backend llm (giữ nguyên comment không cần thiết)
import yaml
cfg = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
cfg['extract']['backend'] = 'llm'
cfg['extract']['lora_adapter'] = ADAPTER
yaml.safe_dump(cfg, open('configs/config.yaml', 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('extract.lora_adapter =', cfg['extract']['lora_adapter'])

extract.lora_adapter = /content/drive/MyDrive/models/qwen7b-lora


In [8]:
# 8) Chạy pipeline (QLoRA) trên 30 file dev để so với few-shot, rồi đo
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
!python src/pipeline.py --input devset/input --output dev_pred --backend llm
!python src/eval/scorer.py --pred dev_pred --gold data/dev/gold --mode overlap
!python src/eval/scorer.py --pred dev_pred --gold data/dev/gold --mode exact

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 30 file | out=dev_pred
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:25<1:23:13, 14.82s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [02:18<00:00,  2.44it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/peft/config.py", line 317, in _get_peft_type
    config_file = hf_hub_download(
                  ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 84, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py",

In [9]:
# 9) (Khi đã hài lòng) chạy full 100 file + đóng gói + tải về
!python src/pipeline.py --input data/test/input --output output --backend llm
!python scripts/package_submission.py --output output --input data/test/input --n 100
from google.colab import files; files.download('output.zip')

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=53 brands=4150
[pipeline] backend=llm | 100 file | out=output
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/339 [00:24<1:19:09, 14.09s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 339/339 [02:08<00:00,  2.64it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/peft/config.py", line 317, in _get_peft_type
    config_file = hf_hub_download(
                  ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 84, in _inner_fn
    validate_repo_id(arg_value)
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", 

FileNotFoundError: Cannot find file: output.zip